In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── Paths ─────────────────────────────────────────────────────────────────────
LIAR_DIR   = r"C:\Users\Kian\Documents\GitHub\GDSProject\liar_dataset"
MODEL_PATH = r"C:\Users\Kian\Documents\GitHub\GDSProject\fakenews_model_final.pt"
TFIDF_PATH = r"C:\Users\Kian\Documents\GitHub\GDSProject\tfidf_vectorizer.pkl"

# ── Hyperparameters (must match training exactly) ─────────────────────────────
VOCAB_SIZE = 30_000
HIDDEN_1   = 512
HIDDEN_2   = 128
DROPOUT_1  = 0.4
DROPOUT_2  = 0.3
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Model definition (must match training exactly) ────────────────────────────
class FakeNewsNet(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, drop1, drop2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(drop1),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(drop2),

            nn.Linear(hidden2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# ── Load model and vectorizer ─────────────────────────────────────────────────
print("Loading model and vectorizer...")

model = FakeNewsNet(VOCAB_SIZE, HIDDEN_1, HIDDEN_2, DROPOUT_1, DROPOUT_2).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

with open(TFIDF_PATH, "rb") as f:
    tfidf = pickle.load(f)

print("  Done.")

# ── Load LIAR dataset ─────────────────────────────────────────────────────────
LIAR_COLS = [
    "id", "label", "statement", "subject", "speaker",
    "job", "state", "party", "barely_true_c", "false_c",
    "half_true_c", "mostly_true_c", "pants_fire_c", "context"
]

def load_liar(split):
    fname = "valid" if split == "val" else split
    return pd.read_csv(f"{LIAR_DIR}\\{fname}.tsv",
                       sep="\t", header=None, names=LIAR_COLS)

train_df = load_liar("train")
val_df   = load_liar("val")
test_df  = load_liar("test")

print(f"Loaded — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# ── Binarise LIAR labels ──────────────────────────────────────────────────────
REAL_LABELS = {"true", "mostly-true", "half-true"}

def binarize_labels(series):
    return (series.str.strip().str.lower().isin(REAL_LABELS)).astype(int)

# ── Inference function ────────────────────────────────────────────────────────
def predict_df(df):
    """Run the FakeNewsCorpus model on a LIAR dataframe."""
    texts = df["statement"].fillna("").str.lower().str.strip()
    X     = tfidf.transform(texts)          # use transform, never fit_transform
    labels = binarize_labels(df["label"])

    all_probs, all_preds = [], []

    # Process in batches to avoid memory issues
    batch_size = 512
    for i in range(0, X.shape[0], batch_size):
        X_batch = X[i:i+batch_size].toarray()
        x_t     = torch.tensor(X_batch, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            probs = model(x_t).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend((probs >= 0.5).astype(int))

    return np.array(labels), np.array(all_preds), np.array(all_probs)

# ── Evaluate on all splits ────────────────────────────────────────────────────
def evaluate(df, split_name):
    labels, preds, probs = predict_df(df)
    auc = roc_auc_score(labels, probs)

    print(f"\n{'═'*60}")
    print(f"  {split_name} — FakeNewsCorpus model on LIAR")
    print(f"{'═'*60}")
    print(classification_report(labels, preds,
                                 target_names=["fake", "real"], digits=4))
    print(f"  ROC-AUC: {auc:.4f}")
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(labels, preds)
    print(f"              fake    real")
    print(f"  actual fake {cm[0,0]:>7,} {cm[0,1]:>7,}")
    print(f"  actual real {cm[1,0]:>7,} {cm[1,1]:>7,}")

evaluate(train_df, "Train")
evaluate(val_df,   "Validation")
evaluate(test_df,  "Test")

Loading model and vectorizer...
  Done.
Loaded — Train: 10,240 | Val: 1,284 | Test: 1,267

════════════════════════════════════════════════════════════
  Train — FakeNewsCorpus model on LIAR
════════════════════════════════════════════════════════════
              precision    recall  f1-score   support

        fake     0.4389    0.9922    0.6086      4488
        real     0.6316    0.0104    0.0205      5752

    accuracy                         0.4407     10240
   macro avg     0.5353    0.5013    0.3146     10240
weighted avg     0.5471    0.4407    0.2783     10240

  ROC-AUC: 0.5362

  Confusion Matrix (rows=actual, cols=predicted):
              fake    real
  actual fake   4,453      35
  actual real   5,692      60

════════════════════════════════════════════════════════════
  Validation — FakeNewsCorpus model on LIAR
════════════════════════════════════════════════════════════
              precision    recall  f1-score   support

        fake     0.4784    0.9886    0.6448